# 01 · Segmentation Model

**Data:** `uci_customer_features.parquet` (UCI Online Retail II)  
**Algorithms:** KMeans, GMM, Hierarchical (Agglomerative)  
**Business labels:** high value · discount seekers · at risk · new users

Run `python scripts/uci_pipeline.py` first.


In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


def find_project_root() -> Path:
    path = Path.cwd().resolve()
    for candidate in (path, *path.parents):
        if (candidate / "scripts" / "build_datasets.py").exists():
            return candidate
    return path


PROJECT_ROOT = find_project_root()
DATA = PROJECT_ROOT / "data" / "modeling"
MODELS = PROJECT_ROOT / "models"
MODELS.mkdir(exist_ok=True)


def load_parquet(name: str) -> pd.DataFrame:
    path = DATA / name
    if not path.exists():
        raise FileNotFoundError(f"Missing {path} — run: python scripts/uci_pipeline.py")
    return pd.read_parquet(path)


def save_artifact(name: str, obj) -> Path:
    path = MODELS / name
    joblib.dump(obj, path)
    print(f"Saved → {path.relative_to(PROJECT_ROOT)}")
    return path


def audit_and_clean(
    df: pd.DataFrame,
    *,
    subset: list[str] | None = None,
    id_col: str | None = None,
    required_cols: list[str] | None = None,
    label: str = "dataset",
) -> pd.DataFrame:
    """Report and drop duplicate rows + rows with NA in required columns (before split)."""
    out = df.copy()
    n0 = len(out)
    dup_subset = subset if subset is not None else ([id_col] if id_col else None)
    n_dup = out.duplicated(subset=dup_subset, keep="first").sum() if dup_subset else out.duplicated(keep="first").sum()
    if n_dup:
        out = out.drop_duplicates(subset=dup_subset, keep="first")
    req = [c for c in (required_cols or []) if c in out.columns]
    na_rows = out[req].isna().any(axis=1).sum() if req else 0
    na_by_col = out[req].isna().sum()
    if req:
        out = out.dropna(subset=req)
    print(
        f"[{label}] {n0:,} rows -> {len(out):,} | "
        f"dropped {n_dup:,} duplicates, {na_rows:,} rows with NA"
    )
    if na_rows and (na_by_col > 0).any():
        print("  NA counts:", na_by_col[na_by_col > 0].to_dict())
    return out


In [ ]:
feat = load_parquet("uci_customer_features.parquet")

CLUSTER_FEATURES = [
    "R_score", "F_score", "M_score", "RFM_score",
    "avg_basket_size", "discount_dependency", "engagement_score",
    "churn_inertia_score", "recency_days", "total_orders",
]
feat = audit_and_clean(
    feat,
    id_col="customer_id",
    required_cols=["customer_id", *CLUSTER_FEATURES],
    label="customer_features",
)
X_raw = feat[CLUSTER_FEATURES]
print(f"Customers: {len(X_raw):,} · Features: {len(CLUSTER_FEATURES)}")
X_raw.describe().T.round(2)


In [ ]:
# Correlation & feature pruning (|r| > 0.85)
corr = X_raw.corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Feature correlation — segmentation inputs")
plt.tight_layout()
plt.show()

upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
drop_cols = [c for c in upper.columns if any(upper[c].abs() > 0.85)]
SELECTED = [c for c in CLUSTER_FEATURES if c not in drop_cols]
print("Dropped (high collinearity):", drop_cols)
print("Selected:", SELECTED)
X = X_raw[SELECTED]


In [ ]:
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_hold = train_test_split(X, test_size=0.30, random_state=RANDOM_STATE)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_hold_s = scaler.transform(X_hold)

K = 4  # business segments
candidates = {}

km = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=20)
km_labels = km.fit_predict(X_train_s)
candidates["KMeans"] = {
    "model": km, "train_labels": km_labels,
    "silhouette": silhouette_score(X_train_s, km_labels),
    "davies_bouldin": davies_bouldin_score(X_train_s, km_labels),
    "calinski": calinski_harabasz_score(X_train_s, km_labels),
}

gmm = GaussianMixture(n_components=K, random_state=RANDOM_STATE, n_init=5)
gmm_labels = gmm.fit_predict(X_train_s)
candidates["GMM"] = {
    "model": gmm, "train_labels": gmm_labels,
    "silhouette": silhouette_score(X_train_s, gmm_labels),
    "davies_bouldin": davies_bouldin_score(X_train_s, gmm_labels),
    "calinski": calinski_harabasz_score(X_train_s, gmm_labels),
}

agg = AgglomerativeClustering(n_clusters=K)
agg_labels = agg.fit_predict(X_train_s)
candidates["Hierarchical"] = {
    "model": agg, "train_labels": agg_labels,
    "silhouette": silhouette_score(X_train_s, agg_labels),
    "davies_bouldin": davies_bouldin_score(X_train_s, agg_labels),
    "calinski": calinski_harabasz_score(X_train_s, agg_labels),
}

metrics = pd.DataFrame({
    name: {k: v for k, v in d.items() if k != "model" and k != "train_labels"}
    for name, d in candidates.items()
}).T.round(4)
display(metrics)
best_name = metrics["silhouette"].idxmax()
print(f"Best by silhouette: {best_name}")


In [ ]:
# Map clusters → business segments (rule-based on cluster centroids)
best = candidates[best_name]
centroids = pd.DataFrame(scaler.inverse_transform(
    getattr(best["model"], "cluster_centers_", None)
    if best_name == "KMeans"
    else pd.DataFrame(X_train_s).groupby(best["train_labels"]).mean().values
), columns=SELECTED)
centroids["cluster"] = range(len(centroids))

def label_cluster(row) -> str:
    if row["R_score"] >= 4 and row["M_score"] >= 4:
        return "high value"
    if row["discount_dependency"] >= centroids["discount_dependency"].median():
        return "discount seekers"
    if row["recency_days"] if "recency_days" in row.index else row.get("churn_inertia_score", 0) >= centroids.get("churn_inertia_score", pd.Series([1])).median():
        return "at risk"
    if row["total_orders"] <= centroids["total_orders"].quantile(0.25):
        return "new users"
    return "core"

# Apply best model to full dataset
X_full_s = scaler.transform(X)
if best_name == "KMeans":
    labels = best["model"].predict(X_full_s)
elif best_name == "GMM":
    labels = best["model"].predict(X_full_s)
else:
    labels = best["model"].fit_predict(X_full_s)

feat_out = feat.copy()
feat_out["cluster_id"] = labels
centroid_df = feat_out.groupby("cluster_id")[SELECTED].mean()
business_map = {}
for cid, row in centroid_df.iterrows():
    if row["M_score"] >= 4 and row["F_score"] >= 3:
        business_map[cid] = "high value"
    elif row["discount_dependency"] >= feat_out["discount_dependency"].median():
        business_map[cid] = "discount seekers"
    elif row["recency_days"] >= 120 or row["churn_inertia_score"] >= 1.2:
        business_map[cid] = "at risk"
    elif row["total_orders"] <= feat_out["total_orders"].quantile(0.30):
        business_map[cid] = "new users"
    else:
        business_map[cid] = "core"
feat_out["segment_label"] = feat_out["cluster_id"].map(business_map)
display(feat_out["segment_label"].value_counts())

artifact = {
    "model_name": best_name,
    "model": best["model"],
    "scaler": scaler,
    "features": SELECTED,
    "business_map": business_map,
    "metrics": metrics.loc[best_name].to_dict(),
}
save_artifact("01_segmentation_best.joblib", artifact)
